In [2]:
%load_ext autoreload
%autoreload 2


In [3]:
import os
import sys
sys.path.append(os.path.abspath(os.path.join("..")))

import pandas as pd
from src.data_loader import load_and_merge_data, merge_oil_data
from src.features import build_macro_features, build_time_features, build_lag_features
from src.train import run_baseline_training

# 1. Pipeline execution including our history features
df = load_and_merge_data()
df = merge_oil_data(df)
df = build_time_features(df)
df = build_lag_features(df)    # <-- Injected history lags
df = build_macro_features(df)

# 2. Track this run in MLflow as a distinct experiment comparison step
model, features = run_baseline_training(df, experiment_name="added_historical_lags")


⏳ Loading train data...
🗓️ Repairing missing timeline gaps...
⏳ Loading store metadata...
🔗 Merging datasets...
% Optimizing memory usage...
✓ Data loaded successfully! Final shape: (3008016, 10)
⏳ Ingesting and treating oil prices...
✓ Oil price feature integrated cleanly.
🗓️ Engineering calendar and payday features...
⏳ Engineering structural historical lags...
✓ Time-series memory features generated cleanly.
🛢️ Engineering advanced oil price trends...
✂️ Splitting data into Train and Validation sets...
📊 Tracking execution under run: 'added_historical_lags'
🚀 Training LightGBM model...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.121788 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2149
[LightGBM] [Info] Number of data points in the train set: 2970594, number of used features: 21
[LightGBM] [Info] Start training from score 2.910

In [45]:
import os
import sys
sys.path.append(os.path.abspath(os.path.join("..")))

import pandas as pd
from src.data_loader import load_and_merge_data, merge_oil_data, merge_holiday_data
from src.features import build_time_features, build_lag_features, build_macro_features, build_promotion_features
from src.train import run_adaptive_scale_training

# Run clean data pipelines
df = load_and_merge_data()
df = merge_oil_data(df)
df = merge_holiday_data(df)
df = build_time_features(df)
df = build_lag_features(df)      # Strict rhythm matching
df = build_promotion_features(df)
df = build_macro_features(df)

# Execute the updated 1D tracking loop script
family_models, family_scale_types, features = run_adaptive_scale_training(df, experiment_name="adaptive_per_family_scaling")


⏳ Loading train data...
🗓️ Repairing missing timeline gaps...
⏳ Loading store metadata...
🔗 Merging datasets...
% Optimizing memory usage...
✓ Data loaded successfully! Final shape: (3008016, 10)
⏳ Ingesting and treating oil prices...
✓ Oil price feature integrated cleanly.
⏳ Ingesting and processing true holiday events...
✓ Definitive holiday tracking integrated.
🗓️ Engineering calendar features and cyclical time signals...
⏳ Engineering strict week-matching historical lags (21 and 28 days out)...
✓ Strict day-of-week matching features generated cleanly.
📢 Engineering short-term promotional memory and holiday proximity windows...
🛢️ Engineering advanced oil price trends...
✂️ Preparing Train and Validation horizons...
🚀 Commencing adaptive scale loop training across 33 families...

🎉 Adaptive Scale Validation RMSLE Score: 0.4042


In [50]:
from src.data_loader import prepare_test_inference_data
from src.train import generate_adaptive_scale_submission  # Our new mixed-scale generator

# 1. Prepare clean data grid
inference_df, test_row_count = prepare_test_inference_data(df)

# 2. Process features sequentially across the clean test grid
inference_df = build_time_features(inference_df)
inference_df = build_lag_features(inference_df)
inference_df = build_promotion_features(inference_df)
inference_df = build_macro_features(inference_df)

# 3. Generate your fixed submission file using the adaptive dictionary maps
generate_adaptive_scale_submission(
    inference_df, test_row_count, family_models, family_scale_types, features
)


⏳ Loading raw test dataset...
⏳ Loading store metadata for test set...
🔗 Stitching historical training tail to test grid for lag computations...
🗓️ Engineering calendar features and cyclical time signals...
⏳ Engineering strict week-matching historical lags (21 and 28 days out)...
✓ Strict day-of-week matching features generated cleanly.
📢 Engineering short-term promotional memory and holiday proximity windows...
🛢️ Engineering advanced oil price trends...
🔮 Running adaptive-scale inference loop on future test grid...
📋 Re-aligning adaptive predictions with raw Kaggle test format...
📊 Submission Row Count: 28512 | Expected: 28512
🎉 Pristine adaptive-scale submission file saved successfully to: C:\Users\adis2\Desktop\Python\Kaggle\store-sales-forecasting\data\processed\submission.csv


In [51]:
from src.evaluate import run_adaptive_error_analysis

# Run a post-mortem on your adaptive scale models to see the true error layout
val_analysis_df = run_adaptive_error_analysis(df, family_models, family_scale_types, features)


🔎 Extracting validation slices for adaptive-scale error analysis...

⚠️ --- Top 5 Product Families with Highest Average Error ---
family
BEVERAGES    541.838405
GROCERY I    531.043939
CLEANING     304.793359
PRODUCE      235.096151
DAIRY         84.810690
Name: absolute_error, dtype: float64

🏢 --- Top 5 Stores with Highest Average Error ---
store_nbr
44    153.369966
47    132.925042
40    128.230258
45    119.357625
49    118.298468
Name: absolute_error, dtype: float64
